In [1]:
#@markdown ## <font color="8A2BE2"> **Google Colab Anti-Disconnect.** 🔌
#@markdown ---
#@markdown #### Avoid automatic disconnection. Still, it will disconnect after <font color="orange">**6 to 12 hours**</font>.

import IPython
js_code = '''
function ClickConnect(){
console.log("Working");
document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect,60000)
'''
display(IPython.display.Javascript(js_code))

<IPython.core.display.Javascript object>

In [5]:
#@markdown ## <font color="8A2BE2"> **Check GPU type.** 👁️
#@markdown ---
#@markdown #### A higher capable GPU can lead to faster training speeds. By default, you will have a <font color="orange">**Tesla T4**</font>.
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


# 🎙️ Piper Voice Training Guide
Welcome to the Piper Voice Finetuning notebook! This guide will walk you through the entire process of training a custom text-to-speech voice locally using your own audio data.  

Brought to you by pbanj and Gemini even though it caused more headaches than it solved. This came to be because the official [Piper Notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_multilingual_training_notebook.ipynb) doesnt work anymore in colab as google dropped older python versions. I spent a weekend getting that one to work. After that I decided I'd make a new one that uses the new [Official repo](https://github.com/OHF-Voice/piper1-gpl/) from the start because the old [Rhasspy Repo](https://github.com/rhasspy/piper) was archived. I also merged the testing and exporting into the notebook so you dont have to go and deal with setting up another colab setting.

---

### 🗺️ Notebook Overview
Here is a quick breakdown of what each step in this notebook does:

* **Step 1. Master Setup & Environment** - Prepares your workspace. It installs the necessary system tools, mounts your Google Drive for safe storage, and clones the modified Piper repository.
* **Step 2. Select Your Base Model** - Provides an interactive menu to choose a pre-existing voice. Finetuning an existing voice (transfer learning) is significantly faster than training from scratch.
* **Step 3. Dataset Sanitizer** - Prepares your dataset. It extracts your `.zip` of audio files, formats your text transcripts, looks for any open quotes to prevent issues, and automatically filters out audio clips that are too long to prevent GPU memory crashes.
* **Step 4. Configure & Start Training** - The main event. This cell sets your training parameters and begins the actual learning process. It will automatically save rolling backups of your best progress directly to your Google Drive.
* **Step 5. Export & Test Your Voice** - Once you are happy with the training, this cell converts your best saved checkpoint into a lightweight `.onnx` file, generates a test audio clip so you can hear the results, and packages everything into a neat `.zip` file.

---

### 🧠 Decoding the Training Terminology
When you run Step 4, you'll see a several updating metrics. Here is exactly what the AI is doing under the hood:

* **Batch Size:** This is how many audio files the AI listens to at the exact same time before it pauses to update its internal math. If you have a batch size of 8, it grabs 8 audio files, listens, guesses how to say the text, corrects itself, and then moves on to the next 8. Larger batch sizes train faster and smoother, but take up more GPU memory (VRAM).
* **Step (or Iteration):** One "step" means the AI has finished processing a single batch.
    * *Example:* If you have a dataset of 160 audio files, and a batch size of 8, it will take **20 steps** to get through the whole list (160 ÷ 8 = 20).
* **Epoch:** One complete pass through your *entire* dataset. When the AI has seen every single audio file exactly one time, that equals 1 Epoch. The AI needs to hear the same dataset thousands of times to learn the fine nuances of a voice, which is why epochs usually count up into the thousands.
* **Checkpoint Interval (`checkpoint_epochs`):** This dictates how often the AI writes its current progress to a permanent `.ckpt` file on your Google Drive. If this is set to 50, the AI will loop through your entire dataset 50 times, evaluate its progress, and save a backup of its brain to the disk.
* **Validation Loss (The "Loss" Number):** Think of this as the AI's error rate, or a golf score—the lower the number, the better! It measures how many mistakes the AI makes when trying to perfectly replicate the nuances of your voice dataset. If a new checkpoint has a lower loss number than the previous one, the voice is getting more accurate. This notebook automatically tracks this score and ensures only your top 3 absolute best models are kept. this will prevent data loss in case the runtime crashes or anything.

---   

### 🔄 How to Recover from a Timeout (Disconnect)
Google Colab will automatically disconnect and wipe its temporary hard drive if left idle for too long even with the above workaround. If you return and see an error about missing files or a disconnected runtime, don't panic! Your trained checkpoints are safe in your Google Drive.

Here's the exact steps to resume your training without losing any progress:

1. **Run Cell 1 (Master Setup):** This is mandatory. It remounts your Google Drive and redownloads the essential Piper tools to the new temporary hard drive.
2. **Skip Cell 2 (Select Base Model):** Your base model is already saved from your first session, so you can skip this entirely.
3. **Run Cell 3 (Dataset Sanitizer):** You *must* run this cell again. Your `wavs.zip` is safe on your Drive, but it needs to be unzipped back into the temporary Colab folder so the trainer can hear the audio.
4. **Run Cell 4 (Configure & Start Training):** Once the audio is unzipped, run this cell. It will automatically find your `last.ckpt` backup file in your Drive and pick up the training exactly where it left off!

In [3]:
#@markdown # <font color="8A2BE2"> **1. Master Setup & Environment** 🛠️
#@markdown ---
#@markdown This essential step prepares your Colab environment for Piper model training. It includes:
#@markdown *   **Permission Checks:** Verifies the presence of your Hugging Face token in Colab Secrets for private model access.
#@markdown *   **Google Drive Mount:** Connects your Google Drive for persistent storage of datasets and models.
#@markdown *   **System & Python Dependencies:** Installs all necessary build tools, libraries (like `cmake`, `espeak-ng`), and Python packages (`lightning`, `librosa`, etc.).
#@markdown *   **Piper Repository Setup:** Clones the `OHF-Voice/piper1-gpl` repository, installs its training components, and builds critical Cython extensions.
#@markdown *   **Dynamic Model Fetching:** Sets up an interactive UI using `ipywidgets` to browse and select available Piper models from Hugging Face based on language.

import os
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import userdata, drive
import torch
import sys


# Known "Good" versions for environment stability
REQUIRED_LIGHTNING = "2.1.0"

def check_dependencies():
    print("🔍 Checking environment stability...")

    # Check GPU
    if not torch.cuda.is_available():
        print("❌ WARNING: No GPU detected. Training will be impossibly slow.")
    else:
        print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")

    # Version Enforcement
    print(f"✅ Python {sys.version.split()[0]}")
    print(f"✅ Torch {torch.__version__}")

    # Direct install with pinning to ensure stability
    print("\n📦 Ensuring stable dependencies...")
    !pip install -q lightning=={REQUIRED_LIGHTNING} gradio huggingface_hub librosa pandas tqdm
    print("✅ Dependencies pinned and verified.")

check_dependencies()

# --- 1. EARLY PERMISSIONS CHECK ---
try:
    hf_token = userdata.get('HF_TOKEN')
    print("✅ Hugging Face Token found in Secrets.")
except:
    print("⚠️ HF_TOKEN not found in Secrets. Private models may be inaccessible.")

# Mount Google Drive
drive.mount('/content/drive')

# --- 2. SYSTEM INSTALLS (Quiet Mode) ---
print("📥 Installing dependencies... (This takes ~2 mins)")
!apt-get update -qq && apt-get install -y -qq build-essential cmake ninja-build espeak-ng libsndfile1-dev
# We install ipywidgets specifically for the interactive menus
!pip install -q ipywidgets

# --- 3. REPOSITORY SETUP & SELF-UPDATE ---
%cd /content
if not os.path.exists('piper1-gpl'):
    print("🛰️ Cloning official Piper repository...")
    !git clone -q https://github.com/OHF-Voice/piper1-gpl.git

%cd piper1-gpl

print("📦 Installing/Updating OHF training dependencies...")
!pip install -q -e '.[train]'

# Re-enforce our stability pin just in case the repo tries to pull a broken lightning version
!pip install -q lightning=={REQUIRED_LIGHTNING}

print("🔨 Building critical C++ extensions...")
!bash build_monotonic_align.sh > /dev/null 2>&1
!python3 setup.py build_ext --inplace > /dev/null 2>&1

# --- 4. THE BRAIN: RECURSIVE HF REGISTRY ---
# This function is used by Step 2 to find models
def fetch_available_models(lang_code="en", quality_filter="medium"):
    models = []
    # Using the 'recursive' parameter in the API to get everything at once
    api_url = f"https://huggingface.co/api/datasets/rhasspy/piper-checkpoints/tree/main/{lang_code}?recursive=True"

    try:
        # Check for token to avoid rate limits on the HF API
        token = userdata.get('HF_TOKEN')
        headers = {"Authorization": f"Bearer {token}"}
    except:
        headers = {}

    try:
        r = requests.get(api_url, headers=headers, timeout=10)
        if r.status_code == 200:
            for item in r.json():
                # We only care about .ckpt files
                if item['path'].endswith('.ckpt'):
                    path_parts = item['path'].split('/')

                    # Search for the quality tag anywhere in the path
                    if quality_filter in path_parts:
                        # Extract a nice name (usually the folder before the quality)
                        try:
                            q_idx = path_parts.index(quality_filter)
                            name = path_parts[q_idx - 1]
                        except:
                            name = path_parts[1]

                        filename = path_parts[-1].replace(".ckpt", "")
                        display_epoch = filename.split('-')[0].replace("epoch=", "Epoch ")

                        label = f"{name} - {display_epoch}"
                        url = f"https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/{item['path']}"
                        models.append((label, url))
    except Exception as e:
        print(f"📡 Connection Error: {e}")

    return sorted(list(set(models))) # Set removes duplicates

clear_output()
print("✅ ENVIRONMENT READY")
print("✅ PERMISSIONS GRANTED")
print("✅ REPOSITORY CLONED & BUILT")

✅ ENVIRONMENT READY
✅ PERMISSIONS GRANTED
✅ REPOSITORY CLONED & BUILT


 If you are using a custom or restricted base model from Hugging Face, make sure to add your HF_TOKEN to the Colab Secrets (the 🔑 icon on the left). This allows the notebook to securely fetch your models without you having to hardcode your password.

In [12]:
#@markdown # <font color="8A2BE2"> **2. Select Your Base Model** 🌍
#@markdown ---
# @markdown > **SETUP:** Add your Hugging Face token to Colab "Secrets" (🔑 icon on the left) named `HF_TOKEN` for full access.

import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# Define the common layout for alignment
item_layout = widgets.Layout(width='400px')
style = {'description_width': '100px'}

# UI Elements
language_input = widgets.Dropdown(
    options=['ar', 'bg', 'ca', 'cs', 'da', 'de', 'el', 'en', 'es', 'fi', 'fr', 'hi', 'hu', 'id', 'it', 'ka', 'ku', 'lb', 'ml', 'ne', 'nl', 'no', 'pl', 'pt', 'ro', 'ru', 'sk', 'sr', 'sw', 'te', 'tr', 'uk', 'ur', 'vi', 'zh'],
    value='en', description='1. Language:', style=style, layout=item_layout
)

quality_input = widgets.Dropdown(
    options=['x-low', 'low', 'medium', 'high'],
    value='medium', description='2. Quality:', style=style, layout=item_layout
)

# Purple Button matching your headers
scan_button = widgets.Button(
    description="🔎 Scan for Voices",
    layout=widgets.Layout(width='395px', margin='0px 0px 10px 105px') # Aligned with the dropdown start
)
scan_button.style.button_color = '#8A2BE2'
scan_button.style.font_weight = 'bold'

download_button = widgets.Button(
    description="⬇️ Download Selected Model",
    layout=widgets.Layout(width='395px', margin='10px 0px 10px 105px')
)
download_button.style.button_color = '#8A2BE2'
download_button.style.font_weight = 'bold'

voice_select_widget = widgets.Dropdown(
    options=[('Click Scan to start...', None)],
    description='3. Base Voice:', style=style, layout=item_layout
)

output = widgets.Output()

def perform_scan(b=None):
    with output:
        voice_select_widget.options = [('🔎 Scanning...', None)]
        results = fetch_available_models(language_input.value, quality_input.value)
        if results:
            voice_select_widget.options = results
            print(f"✅ Found {len(results)} models.")
        else:
            voice_select_widget.options = [('❌ No models found', None)]
            print("⚠️ Try a different quality or check your HF_TOKEN.")
        clear_output(wait=True)

def download_model(b=None):
    with output:
        clear_output()
        target_url = custom_url if custom_url else voice_select_widget.value
        if target_url:
            print(f"⬇️ Downloading base model from {target_url}...")
            !wget -q --show-progress -O /content/base_model.ckpt "{target_url}"
            if os.path.exists("/content/base_model.ckpt"):
                print("✅ Download complete! You can now proceed to Step 3.")
            else:
                print("❌ Download failed. Check the URL.")
        else:
            print("⚠️ No model selected or URL provided.")

scan_button.on_click(perform_scan)
download_button.on_click(download_model)

# Layout
ui = widgets.VBox([scan_button, language_input, quality_input, voice_select_widget, download_button])
display(ui, output)

# Run initial scan automatically
perform_scan()

# @markdown ---
# @markdown **Manual Override**
custom_url = "" # @param {type:"string"}

Output()

In [ ]:
#@markdown # <font color="8A2BE2"> **3. Dataset Sanitizer (Audio & Text)** 🛠️
#@markdown ---
import os
import shutil
import glob
import pandas as pd
import librosa
import zipfile
from tqdm import tqdm

# @markdown ### 📂 Input Paths
# @markdown Tip: You can right-click files in the left sidebar and select "Copy Path".
dataset_zip = "/content/drive/MyDrive/wavs.zip" # @param {type:"string"}
transcript_path = "/content/drive/MyDrive/bender_metadata.txt" # @param {type:"string"}

# @markdown ### 🧪 VRAM Protection
max_seconds = 10 # @param [5, 8, 10, 12, 15]

# --- Logic ---
audio_dir = "/content/temp_audio"

# 1. Extraction with Check
if os.path.exists(dataset_zip):
    with zipfile.ZipFile(dataset_zip, 'r') as z:
        z.extractall(audio_dir)
    print(f"📦 Audio extracted to {audio_dir}")
else:
    raise FileNotFoundError(f"❌ Could not find {dataset_zip}. Double check your path!")

# 2. Smart Delimiter Sniffing
def clean_transcript(path):
    with open(path, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        delim = "|" if "|" in first_line else ","

    df = pd.read_csv(path, sep=delim, header=None, names=['file', 'text'])
    df['text'] = df['text'].str.replace('"', '', regex=False).str.strip()
    df['file'] = df['file'].str.strip()
    return df

df = clean_transcript(transcript_path)

# 3. Duration Filtering (Smart Path Version)
print(f"⏱️ Filtering audio (Max: {max_seconds}s)...")
valid_files = []

# Find all .wav files in the directory recursively
# This maps '001.wav' to '/content/temp_audio/dataset/wav/001.wav' automatically
audio_map = {}
for root, dirs, files in os.walk(audio_dir):
    for filename in files:
        if filename.endswith(".wav"):
            audio_map[filename] = os.path.join(root, filename)

for index, row in tqdm(df.iterrows(), total=len(df)):
    file_key = row['file']
    if not file_key.endswith('.wav'): file_key += '.wav'

    # Check our map for the file
    if file_key in audio_map:
        full_path = audio_map[file_key]
        if librosa.get_duration(path=full_path) <= max_seconds:
            # Update the row to use the relative path inside temp_audio
            # This makes sure the trainer can find it later
            row['file'] = os.path.relpath(full_path, audio_dir)
            valid_files.append(row)

# 4. Final Metadata Export
output_dir = "/content/training_data"
os.makedirs(output_dir, exist_ok=True)
final_df = pd.DataFrame(valid_files)
final_df.to_csv(f"{output_dir}/metadata.csv", sep="|", header=False, index=False)

print(f"\n✅ SUCCESS: {len(final_df)} files ready for training.")

In [ ]:
#@markdown # <font color="8A2BE2"> **3.5 Monitor Training (Live Graphs)** 📊
#@markdown ---
#@markdown Run this cell to open TensorBoard. It will show you the training loss and other metrics in real-time.
#@markdown

%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/piper_training/{voice_name}/lightning_logs"

In [ ]:
#@markdown # <font color="8A2BE2"> **4. Configure & Start Training** ⚙️
#@markdown ---
# @markdown > **TIP:** If training stops or crashes, simply set the **action** to "Continue training" and run this cell again.
# @markdown ---
# @markdown > **MULTI-SPEAKER WARNING:** To use "convert to multi-speaker", your metadata.csv MUST have 3 columns: `file|speaker_id|text`. If you only have `file|text`, use "finetune".


import os, sys, shutil, glob


# --- 1. SYSTEM PURGE & PATH SETUP ---
print("🧹 Purging conflicting installs...")
!pip uninstall -y piper piper-tts > /dev/null 2>&1

repo_root = "/content/piper1-gpl"
src_path = os.path.join(repo_root, "src")
piper_path = os.path.join(src_path, "piper")

if src_path not in sys.path: sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = f"{src_path}:" + os.environ.get('PYTHONPATH', '')

# --- 2. FINALIZE C++ EXTENSIONS ---
print("🔨 Finalizing C++ extensions...")
%cd {repo_root}
!python3 setup.py build_ext --inplace > /dev/null 2>&1

compiled_so = glob.glob(os.path.join(src_path, "piper", "espeakbridge*.so"))
if compiled_so:
    dest = os.path.join(src_path, "piper", "espeakbridge.so")
    shutil.copy2(compiled_so[0], dest)
    print(f"✅ Bridge locked: {os.path.basename(compiled_so[0])}")

# --- 3. PYTORCH 2.6 COMPATIBILITY PATCH ---
os.makedirs(piper_path, exist_ok=True)
piper_init = os.path.join(piper_path, "__init__.py")
init_content = ""
if os.path.exists(piper_init):
    with open(piper_init, "r") as f: init_content = f.read()

if "add_safe_globals" not in init_content:
    print("🩹 Applying PyTorch 2.6 compatibility patch...")
    patch = "import torch\nimport pathlib\nif hasattr(torch.serialization, 'add_safe_globals'):\n    torch.serialization.add_safe_globals([pathlib.PosixPath, pathlib.WindowsPath])\n\n"
    with open(piper_init, "w") as f: f.write(patch + init_content)

# --- 4. DYNAMIC UI SETTINGS ---
# @markdown ### 📂 Project Paths
voice_name = "bonder" # @param {type:"string"}
# @markdown ### 🚀 Training Action
# @markdown * `finetune`: Best for single-voice training.
# @markdown * `resume`: Pick up exactly where you left off.
# @markdown * `convert`: Convert a single-voice base model into a multi-speaker model.
action = "finetune" # @param ["finetune", "resume", "convert"]

# @markdown ### 🏃 Hyperparameters
batch_size = 8 # @param [4, 8, 12, 16, 24]
max_epochs = 10000 # @param {type:"number"}
checkpoint_epochs = 50 # @param {type:"slider", min:10, max:1000, step:10}

training_dir = f"/content/drive/MyDrive/piper_training/{voice_name}"
metadata_path = "/content/training_data/metadata.csv"
audio_dir = "/content/temp_audio"

# Map UI selections
try:
    lang_code = language_input.value
    quality_tier = quality_input.value
except:
    lang_code, quality_tier = "en", "medium"

sample_rate = 22050 if quality_tier != "x-low" else 16000
espeak_lang = {'en': 'en-us', 'zh': 'cmn', 'pt': 'pt-br'}.get(lang_code, lang_code)
config_path = f"{repo_root}/etc/vits/{lang_code}_US/{quality_tier}/config.json"

# --- 5. ACTION LOGIC ---
latest_ckpt = "/content/base_model.ckpt"

if action == "resume":
    ckpts = glob.glob(f"{training_dir}/lightning_logs/version_*/checkpoints/*.ckpt")
    if ckpts:
        latest_ckpt = max(ckpts, key=os.path.getctime)
        print(f"🔄 RESUME: Found {os.path.basename(latest_ckpt)}")
    else:
        print("⚠️ RESUME: No checkpoints found. Falling back to base model.")

# Multi-speaker conversion logic
action_arg = "fit"
if action == "convert":
    action_arg = "convert_to_multi_speaker"
    print("👥 MULTI-SPEAKER: Converting base model to multi-speaker...")

ckpt_cmd = f'--ckpt_path "{latest_ckpt}"' if os.path.exists(latest_ckpt) else ""

# --- 6. LAUNCH TRAINING ---
%cd {repo_root}
print(f"\n🚀 PVC-Pipe: Starting {voice_name} (Action: {action})")

!python3 -m piper.train {action_arg} \
  --data.voice_name "{voice_name}" \
  --data.csv_path "{metadata_path}" \
  --data.audio_dir "{audio_dir}" \
  --data.cache_dir "/content/cache" \
  --data.espeak_voice "{espeak_lang}" \
  --data.config_path "{config_path}" \
  --model.sample_rate {sample_rate} \
  --data.batch_size {batch_size} \
  --trainer.devices 1 \
  --trainer.max_epochs {max_epochs} \
  --trainer.precision "16-mixed" \
  --trainer.default_root_dir "{training_dir}" \
  --trainer.check_val_every_n_epoch {checkpoint_epochs} \
  --trainer.callbacks.save_top_k 3 \
  --trainer.callbacks.monitor "val_loss" \
  --trainer.callbacks.mode "min" \
  --trainer.callbacks.save_last true \
  {ckpt_cmd}

In [ ]:
#@markdown # <font color="8A2BE2"> **5. Export & Test Your Voice** 🎙️
#@markdown ---
#@markdown Run this cell to load an interactive menu where you can select your best checkpoint, convert it to ONNX,  test it out, and download the zip containing the files for it.

import os
import shutil
import glob
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output

# Ensure we have the voice name from Step 4
try:
    voice_name
except NameError:
    voice_name = "bonder" # Fallback if variables were cleared

# --- Path Logic ---
# Pointing directly to your Google Drive where the files actually saved
model_dir = f"/content/drive/MyDrive/piper_training/{voice_name}"
export_dir = "/content/export"
os.makedirs(export_dir, exist_ok=True)

# 1. Scan for the latest checkpoints
ckpt_options = []
version_dirs = glob.glob(f"{model_dir}/lightning_logs/version_*")

if version_dirs:
    # Grab the most recently created version folder
    latest_version = max(version_dirs, key=os.path.getctime)
    ckpts = glob.glob(f"{latest_version}/checkpoints/*.ckpt")

    for ckpt in ckpts:
        # Extract just the filename to make the dropdown look clean
        filename = os.path.basename(ckpt)
        ckpt_options.append((filename, ckpt))

if not ckpt_options:
    ckpt_options = [("❌ No checkpoints found in Drive!", None)]
else:
    # Sort them so last.ckpt is easy to find
    ckpt_options.sort(reverse=True)

# --- 2. Build the Interactive UI ---
style = {'description_width': '150px'}
layout = widgets.Layout(width='600px')

ckpt_dropdown = widgets.Dropdown(
    options=ckpt_options,
    description='1. Select Checkpoint:',
    style=style, layout=layout
)

test_sentence_input = widgets.Textarea(
    value="I am a high-quality local voice, running entirely on open-weight hardware. No cloud required, meatbag!",
    description='2. Test Sentence:',
    style=style, layout=layout
)

export_button = widgets.Button(
    description="⚙️ 1. Export & Test Voice",
    layout=widgets.Layout(width='445px', margin='15px 0px 5px 155px')
)
export_button.style.button_color = '#8A2BE2'
export_button.style.font_weight = 'bold'

download_button = widgets.Button(
    description="🎁 2. Download Zip Package",
    layout=widgets.Layout(width='445px', margin='5px 0px 10px 155px'),
    disabled=True # Stays locked until export finishes
)
download_button.style.button_color = '#20B2AA'
download_button.style.font_weight = 'bold'

output_area = widgets.Output()

# --- 3. Execution Logic ---
def run_export(b):
    with output_area:
        clear_output()
        # Lock the download button while processing
        download_button.disabled = True

        selected_ckpt = ckpt_dropdown.value

        if not selected_ckpt:
            print("❌ ERROR: No valid checkpoint selected.")
            return

        filename = os.path.basename(selected_ckpt)
        print(f"📦 Converting {filename} to ONNX...")

        # Run the Piper ONNX exporter
        !python3 -m piper_train.export_onnx "{selected_ckpt}" "{export_dir}/voice.onnx" > /dev/null 2>&1

        # Locate and copy the config.json
        config_src = None
        if os.path.exists(f"{model_dir}/config.json"):
            config_src = f"{model_dir}/config.json"
        elif os.path.exists("/content/training_data/config.json"):
             config_src = "/content/training_data/config.json"

        if config_src:
            shutil.copy(config_src, f"{export_dir}/voice.onnx.json")
            print(f"✅ Exported to {export_dir}")
        else:
            print(f"⚠️ Warning: Could not find config.json automatically.")

        # Interactive Test
        print("\n🔊 Generating test audio...")
        test_text = test_sentence_input.value
        !echo "{test_text}" | /usr/bin/piper --model "{export_dir}/voice.onnx" --output_file "{export_dir}/test.wav" > /dev/null 2>&1

        if os.path.exists(f"{export_dir}/test.wav"):
            display(Audio(f"{export_dir}/test.wav", autoplay=True))
            print("✨ SUCCESS! Audio generated.")
        else:
            print("❌ ERROR: Audio generation failed.")

        # Create the Zip File silently
        zip_name = f"/content/{voice_name}_piper_model.zip"
        !zip -j -q {zip_name} {export_dir}/voice.onnx {export_dir}/voice.onnx.json

        if os.path.exists(zip_name):
            # Unlock the download button!
            download_button.disabled = False
            print(f"\n🎁 Package ready! Click the Download button above to save.")

def trigger_download(b):
    with output_area:
        zip_name = f"/content/{voice_name}_piper_model.zip"
        if os.path.exists(zip_name):
            from google.colab import files
            print(f"⬇️ Requesting download for {os.path.basename(zip_name)}...")
            files.download(zip_name)
        else:
            print("❌ ERROR: Zip file not found. Please run the export first.")

export_button.on_click(run_export)
download_button.on_click(trigger_download)

# Display everything
display(widgets.VBox([ckpt_dropdown, test_sentence_input, export_button, download_button, output_area]))